# 14 — Cross-cohort stability: the article's figures

Figure 3 and Supplementary Figs. Sa–Sc for **one state set**, chosen in the next cell (HMM,
mean-shift or threshold; `tools/stability_sweep.py` compares them all). The last cell gives the
numbers the text cites.

* **Figure 3.** A.1 reference transition graph (every cohort and condition, each once) · A.2 movie
  vs rest in the same people · B every cohort (occupancy, p(j | i), dynamic metrics) · C the
  transition structure replicates across cohorts and conditions (r over the noise ceiling; with
  `BALANCE = "time"` every cohort contributes the same hours of scan per draw) · D which properties
  depend on context (|d| movie vs rest in the same people; < 0.2 stable, 0.2–0.5 moderate,
  > 0.5 context-dependent).
* **Sa–Sc.** Variation between films, film by film: dynamic metrics, occupancy, transitions.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from fmri_decomposition.unsupervised_model import cohort_tables as ct
from fmri_decomposition.figures import cohorts as cp

# ---- THE STATE SET: any atlas x window x clusterer stage 4b wrote (listed below)
ATLAS    = "harvardoxford"
WINDOW_S = 30.0
STATES   = "HMM2_umap3_8"          # e.g. "HMM2_pca3_8", "ThresholdCluster_umap3_8", "MeanShift_umap3_3"

# ---- panel C: data balance. "time" = every cohort the same HOURS of scan per draw (whole scans,
# pooled transition counts): people and stimulus duration balanced together. None = cohort means.
BALANCE  = "time"
HOURS    = 10
N_BOOT   = 200                     # draws for panel C, bootstrap over people for panel D
SAVE_DIR = None                    # e.g. "figures/14": PNG + SVG of every figure

cfg = ct.Settings(
    space=dict(atlas=ATLAS, window_s=WINDOW_S, states=STATES),
    entropy_estimator="miller_madow",
    cohorts={"ds002837": dict(dirs=["ds002837"], role="train"),
             "cneuromod": dict(dirs=["cneuromod"], role="train"),
             "camcan": dict(dirs=["camcan", "camcan_rest"], role="projected"),
             "hcp7t_fix": dict(dirs=["hcp7t_fix"], role="projected")},
    rest_pattern=r"rest\d*",          # Rest, REST1..4 (HCP); every other task is movie
    alpha=0.10,                        # BH-FDR (A.2, films)
    fc_task=1.0, fc_dyn=0.25,          # A.2: |log2FC| for the graph and the dynamics
    edge_top=12,                       # A.1/A.2: strongest arrows drawn
    sesoi_d=0.2,
    save_dir=SAVE_DIR,
)
sets = ct.available_sets(cfg.root, cfg.cohorts, cfg.modelling_version)
print(f"{len(sets)} state sets have every cohort; at {ATLAS}, {WINDOW_S:g} s:")
print(", ".join(sets[(sets.atlas == ATLAS) & (sets.window_s.astype(float) == WINDOW_S)]["states"]))
co = ct.Cohorts(cfg)
co.tasks()

## Figure 3

In [ ]:
graphs = co.graphs()
marks, _ = co.task_change()
agree = co.agreement(co.dyn)
st = co.stability(n_boot=N_BOOT)
pairs, ceil = co.replication("movie", balance=BALANCE, hours=HOURS, n_boot=N_BOOT)
cp.fig_figure3(co, graphs, marks, st, pairs, agree)

## Supplementary Figs. Sa–Sc — variation between films

One row per movie task (film, episode, run), grouped by cohort. Filled = the cohort's films share
viewers (HCP, CNeuroMod: each person against their own other films, paired); hollow = each film
had its own viewers (NNDb: descriptive). Red = differs from the cohort's other films (BH-FDR).

In [ ]:
fs = co.film_summary()
DYN = ["transition_fit", *co.dyn]
cp.fig_films_forest(co, fs, DYN, co.rest_means(DYN),
                    title="Supplementary Fig. Sa — dynamic metrics, film by film", name="Sa_films_dynamics")

In [ ]:
cp.fig_films_forest(co, fs, co.occ, co.rest_means(co.occ),
                    title="Supplementary Fig. Sb — occupancy of every state, film by film", name="Sb_films_occupancy",
                    labels={o: f"S{o[4:]} (share of time)" for o in co.occ})

In [ ]:
cp.fig_films_matrices(co, fs)

## Numbers for the text

Panel C (replication over the noise ceiling, 95% CI over draws), and every variable on one |d|
scale with its tier: rest vs movie in the same people; film vs film in the same people (HCP,
CNeuroMod); movie across cohorts (different people, films and scanners: "alike despite", not "why").

In [ ]:
display(pairs.round(3))
co.context_summary(st, fs).round(2)